# The REG701_APPLN table

The table `REG701_APPLN` is a very important "building block" for anyone analysing European patents (EP) with unitary effect inside the PATSTAT EP Register database. To help you navigate the database, it is organised into series:

* The 100 Series: These tables handle general information (like the title of the invention or the names of the inventors).

* The 700 Series: This is a newer section of the database created specifically to store information about the Unitary Patent (UP).

Table `REG701_APPLN` acts as a master list. It contains all the EP application numbers that have been granted "unitary effect."
Think of it as the "index" or "header table" for the entire Unitary Patent module. If you want to study Unitary Patents in this database, you always start here, as it connects all other related data together.

## What is a Unitary Patent?
Under the traditional European patent system, a patent granted by the EPO must be validated and maintained separately in each country where protection is sought. This involves differing national requirements and can result in significant costs, including translations, validation and renewal fees, and administrative and attorney expenses, which increase with the number of countries concerned. By contrast, a **Unitary Patent** is a single patent that provides uniform protection across all participating European Union Member States. It eliminates the need to validate and manage patents separately through multiple national patent offices, as the patent is administered as one indivisible right. In addition, legal disputes relating to a UP are handled by a single judicial system (the Unified Patent Court), rather than having different trials in different countries. In short, it is a "one-stop-shop" patent for Europe, making it cheaper and easier for inventors to protect their ideas across the continent.

Eighteen EU Member States participating in enhanced cooperation have already ratified the relevant Agreements and are currently part of the Unitary Patent system: Austria, Belgium, Bulgaria, Denmark, Estonia, Finland, France, Germany, Italy, Latvia, Lithuania, Luxembourg, Malta, the Netherlands, Portugal, Romania, Slovenia, and Sweden. Additional EU Member States are expected to ratify the UPC Agreement in the coming years. Once fully implemented, the Unitary Patent will allow patent protection to be obtained in up to 25 EU Member States through a single application filed with the European Patent Office (EPO). These additional countries are Cyprus, Czech Republic, Greece, Hungary, Ireland, Poland, and Slovakia.

In [1]:
from epo.tipdata.patstat import PatstatClient
from epo.tipdata.patstat.database.models import REG701_APPLN, REG101_APPLN
from sqlalchemy import func

# Initialise the PATSTAT client
patstat = PatstatClient(env='PROD')

# Access ORM
db = patstat.orm()

### ID (primary key)
A technical identifier for an application that has entered the **UP phase**, without business meaning. Its values will not change from one PATSTAT edition to the next.

In [2]:
i = db.query(
    REG701_APPLN.id
).limit(1000)

df = patstat.df(i)
df

,id
0,20845780
1,18795814
2,17828864
3,23166142
4,20726757
...,...
995,17766652
996,20713816
997,21776497
998,18830234


### STATUS
This field indicates the current legal or procedural status of the Unitary Patent. This attribute is specific to the table `REG701_APPLN` and reflects the state of the patent under the Unitary Patent system.
*   Format: A single-digit number ranging from 1 to 9.
*   Decoding the values: The numeric values serve as codes. To understand the specific meaning of each number (e.g., whether the patent is registered, lapsed, or revoked), reference must be made to the separate lookup table: `REG741_APPLN_STATUS`.

In [3]:
q = (
    db.query(
        REG701_APPLN.status
    )
    .order_by(REG701_APPLN.status)
    .distinct()
)

res = patstat.df(q)
res

,status
0,1
1,4
2,5
3,6
4,7
5,8
6,9


## Linking REG701_APPLN to REG101_APPLN
The `REG101_APPLN` table acts as the central pillar of the entire database. It contains one record for every published EP application, including both direct EP applications and Euro-PCT applications. Because a unitary patent must always originate from an existing EP application, every identifier found in the `REG701_APPLN` table is already included in the `REG101_APPLN` table. In other words, `REG701_APPLN` is a subset of `REG101_APPLN`. This structure allows the `REG101_APPLN` table to be used as the main entry point for retrieving common bibliographic data for all applications, including those that later enter the unitary phase described by the tables in the 700 series.

We can therefore link the two tables using the shared `ID` field.

In [4]:
q = (
    db.query(
        REG701_APPLN.id,
        REG701_APPLN.status,
        REG101_APPLN.appln_filing_date,
        REG101_APPLN.appln_auth,
    )
    .join(
        REG101_APPLN,
        REG101_APPLN.id == REG701_APPLN.id
    )
    .distinct()
    .limit(1000)
)

res = patstat.df(q)
res

,id,status,appln_filing_date,appln_auth
0,17786977,9,2017-10-03,EP
1,17705416,9,2017-02-16,EP
2,20754417,9,2020-07-09,EP
3,17862381,9,2017-10-20,EP
4,22203904,9,2022-10-26,EP
...,...,...,...,...
995,21187788,9,2021-07-26,EP
996,20742335,9,2020-06-13,EP
997,20216221,9,2020-12-21,EP
998,19858358,9,2019-06-26,EP


It is also possible to join REG101_APPLN directly with any other table that contains the ID field, because this identifier is consistent and unique across the entire database. Below is an example that shows how this type of connection can be made:

In [5]:
from epo.tipdata.patstat.database.models import REG102_PAT_PUBLN

q = (
    db.query(
        REG701_APPLN.id,
        REG102_PAT_PUBLN.bulletin_year,
        REG102_PAT_PUBLN.bulletin_nr,
        REG102_PAT_PUBLN.publn_date
    )
    .join(
        REG102_PAT_PUBLN,
        REG102_PAT_PUBLN.id == REG701_APPLN.id
    )
    .distinct()
    .limit(1000)
)

res = patstat.df(q)
res

,id,bulletin_year,bulletin_nr,publn_date
0,18838100,2020,23,2020-06-03
1,21163612,2022,38,2022-09-21
2,17910854,2020,16,2020-04-15
3,20768319,2025,2,2025-01-08
4,20780647,2021,20,2021-05-20
...,...,...,...,...
995,18783043,2019,16,2019-04-18
996,19795873,2019,45,2019-11-07
997,18703605,2023,43,2023-10-25
998,21189065,2024,46,2024-11-13
